# Paneles solares: YOLO por fuente

Tres arquitecturas (`yolov8n.pt`, `yolo11n.pt`, `yolo26n.pt`), tres estrategias de congelamiento y tres folds. Son 27 entrenamientos. La métrica principal es el promedio macro de mAP50-95, con el mismo peso para cada fold.

Elegí una GPU NVIDIA en *Entorno de ejecución*. Los pesos `best.pt` se copian a `RESULTS_DIR`, que tiene que vivir en Google Drive.

Las celdas de rutas, instalación y verificación no cargan modelos. La última celda sí entrena.

In [2]:
!git clone https://github.com/ana-castillo-zuain/procesamiento-de-imagenes.git /content/proyecto

Cloning into '/content/proyecto'...
remote: Enumerating objects: 2342, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 2342 (delta 14), reused 87 (delta 7), pack-reused 2248 (from 1)
Receiving objects: 100% (2342/2342), 148.55 MiB | 35.37 MiB/s, done.
Resolving deltas: 100% (40/40), done.
Updating files: 100% (9974/9974), done.


In [3]:
import os

os.environ["PYTHONHASHSEED"] = "42"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Fuera de Colab: no se montó Drive.")

Mounted at /content/drive


In [4]:
from pathlib import Path

# Editar estas tres rutas. No hay una carpeta de Drive fija en el código.
if IN_COLAB:
    REPO_DIR = Path("/content/proyecto")
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "YOLO" else Path.cwd()

CODE_DIR = REPO_DIR / "YOLO"
DATASET_DIR = REPO_DIR / "data" / "yolo"
RESULTS_DIR = Path("/content/drive/MyDrive/procesamiento-de-imagenes/results")

assert CODE_DIR.joinpath("train.py").is_file(), f"No está train.py en {CODE_DIR}"
assert DATASET_DIR.joinpath("PV01_val").is_dir(), f"No está el dataset en {DATASET_DIR}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("código   :", CODE_DIR)
print("dataset  :", DATASET_DIR)
print("resultados:", RESULTS_DIR)

código   : /content/proyecto/YOLO
dataset  : /content/proyecto/data/yolo
resultados: /content/drive/MyDrive/procesamiento-de-imagenes/results


In [5]:
import sys

# No reinstala torch: Colab ya trae la build con CUDA.
!{sys.executable} -m pip install -q -r "{CODE_DIR / 'requirements.txt'}"

In [6]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA no está disponible. Activá una GPU en el entorno de ejecución antes de entrenar. "
        "No se cargó ningún modelo."
    )
print(torch.cuda.get_device_name(0))
print("torch", torch.__version__, "cuda", torch.version.cuda)

Tesla T4
torch 2.11.0+cu130 cuda 13.0


In [7]:
import sys
sys.path.insert(0, str(CODE_DIR))

from solar_yolo.config import Config
from solar_yolo.engine import verify_setup

config = Config(dataset_dir=DATASET_DIR, results_dir=RESULTS_DIR, device=0, batch=16)
print(verify_setup(config))

Estrategias de congelamiento

El argumento freeze de Ultralytics usa el índice top-level del YAML.
freeze=[0, 1, 2] congela esas capas. freeze=10 congela 0..9, no la capa 10.

full: Full fine-tuning. Se entrenan todas las capas. Ultralytics igual congela los parámetros cuyo nombre contiene .dfl cuando existen.
backbone: Backbone congelado. Se congelan todos los módulos del backbone y se entrenan el cuello y la cabeza. El corte es el último módulo declarado bajo backbone en el YAML de esa arquitectura.
partial: Fine-tuning parcial. Se congelan solo el stem y el estadio P2 (dos convoluciones stride 2 y el primer bloque CSP). Se entrena desde el downsample a P3/8 inclusive, junto con el resto del backbone, el cuello y la cabeza.

## yolov8n  (yolov8n.pt)
- Backbone de 10 módulos (0-9). El último es SPPF. No existe C2PSA.
- La capa 2 del estadio P2 es C2f, no C3k2.
- La cabeza es Detect con DFL (reg_max=16). Ultralytics deja .dfl congelado siempre.
Capas:
   0  Conv       backbone   stem  

## Entrenar

`None` recorre las 27 combinaciones. Para una sola, pasá una lista.

Si Colab se queda sin memoria, bajá `batch` en la celda anterior y volvé a correr desde ahí. Un experimento ya terminado, con su `best.pt`, no se repite.

In [ ]:
from solar_yolo.engine import run_experiments

SELECTED_ARCHITECTURES = None  # ["yolov8n", "yolo11n", "yolo26n"]
SELECTED_STRATEGIES = None     # ["full", "backbone", "partial"]
SELECTED_FOLDS = None          # [1, 2, 3]

outcome = run_experiments(
    config,
    architectures=SELECTED_ARCHITECTURES,
    strategies=SELECTED_STRATEGIES,
    folds=SELECTED_FOLDS,
)
print("fallos:", len(outcome["failures"]))
for failure in outcome["failures"]:
    print(failure["experiment_id"], failure["error"])

2026-11-03 12:32:13,421 INFO 
Estrategias de congelamiento

El argumento freeze de Ultralytics usa el índice top-level del YAML.
freeze=[0, 1, 2] congela esas capas. freeze=10 congela 0..9, no la capa 10.

full: Full fine-tuning. Se entrenan todas las capas. Ultralytics igual congela los parámetros cuyo nombre contiene .dfl cuando existen.
backbone: Backbone congelado. Se congelan todos los módulos del backbone y se entrenan el cuello y la cabeza. El corte es el último módulo declarado bajo backbone en el YAML de esa arquitectura.
partial: Fine-tuning parcial. Se congelan solo el stem y el estadio P2 (dos convoluciones stride 2 y el primer bloque CSP). Se entrena desde el downsample a P3/8 inclusive, junto con el resto del backbone, el cuello y la cabeza.

## yolov8n  (yolov8n.pt)
- Backbone de 10 módulos (0-9). El último es SPPF. No existe C2PSA.
- La capa 2 del estadio P2 es C2f, no C3k2.
- La cabeza es Detect con DFL (reg_max=16). Ultralytics deja .dfl congelado siempre.
Capas:
   0

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.200 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/procesamiento-de-imagenes/results/datasets/PV01_val.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0

2026-11-03 12:32:27,960 INFO Congelamiento verificado para yolov8n / full.


Using 422 train, 478 val images for fraction=1.0 at imgsz=256
Using 2 dataloader workers
Logging results to /content/drive/MyDrive/procesamiento-de-imagenes/results/runs/yolov8n__full__fold1
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100     0.395G      2.088      3.307      1.684         20        256: 100% ━━━━━━━━━━━━ 27/27 4.5it/s 6.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 4.5it/s 3.3s
                   all        478       3556      0.312      0.283      0.205     0.0808

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100     0.471G       1.79      1.871      1.437         18        256: 100% ━━━━━━━━━━━━ 27/27 8.2it/s 3.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 3.6it/s 4.2s
                   all        478       

2026-11-03 12:37:32,939 INFO Terminado yolov8n__full__fold1. best.pt en /content/drive/MyDrive/procesamiento-de-imagenes/results/weights/yolov8n/full/fold1/best.pt
2026-11-03 12:37:32,964 INFO Empieza yolov8n__full__fold2. Freeze: (ninguna capa)


Ultralytics 8.4.200 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/procesamiento-de-imagenes/results/datasets/solarpanels_val.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yo

... (122 outputs left out)

2026-11-03 14:28:56,936 INFO Terminado yolo26n__backbone__fold3. best.pt en /content/drive/MyDrive/procesamiento-de-imagenes/results/weights/yolo26n/backbone/fold3/best.pt
2026-11-03 14:28:56,972 INFO Empieza yolo26n__partial__fold1. Freeze: [0, 1, 2]


New https://pypi.org/project/ultralytics/8.4.201 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.200 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/procesamiento-de-imagenes/results/datasets/PV01_val.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=[0, 1, 2], hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_de

2026-11-03 14:28:58,777 INFO Congelamiento verificado para yolo26n / partial.


Using 422 train, 478 val images for fraction=1.0 at imgsz=256
Using 2 dataloader workers
Logging results to /content/drive/MyDrive/procesamiento-de-imagenes/results/runs/yolo26n__partial__fold1
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      1/100     0.441G       1.97      7.016    0.02126         20        256: 100% ━━━━━━━━━━━━ 27/27 5.4it/s 5.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 3.1it/s 4.9s
                   all        478       3556      0.351      0.387        0.3      0.133

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      2/100     0.527G      1.704      3.269    0.01672         18        256: 100% ━━━━━━━━━━━━ 27/27 7.0it/s 3.8s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 5.0it/s 3.0s
                   all        478    